# Projet Kayak

## Extraction des données via API

In [1]:
# initialisation
import pandas as pd
from dotenv import load_dotenv
from pathlib import Path
import boto3
import json
import os
import requests
from time import sleep
from datetime import datetime, date
load_dotenv()

# definition de constantes
USER_AGENT = os.getenv("USER_AGENT")
AWS_ACCESS_KEY = os.getenv("AWS_ACCESS_KEY")
AWS_SECRET_KEY = os.getenv("AWS_SECRET_KEY")
AWS_REGION = os.getenv("AWS_REGION")
AWS_BUCKET = os.getenv("AWS_BUCKET")
AWS_BUCKET_DIR = os.getenv("AWS_BUCKET_DIR")
LOCAL_DIR = "./outputs/"


In [2]:
# configuration base de données
from sqlalchemy import create_engine, text

DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_HOST = os.getenv("DB_HOST")
DB_NAME = os.getenv("DB_NAME")
engine = create_engine(f"postgresql+psycopg://{DB_USER}:{DB_PASSWORD}@{DB_HOST}/{DB_NAME}")

### 1. Les villes

In [3]:
with open('config/cities.json') as f:
    cities = json.load(f)

### 2. Récupération des coordonnées GPS des villes

Utilisation de l'API Nominatim (OpenStreetMap) qui est gratuite et nécessite un user agent.  
Les coordonnées sont mises en cache dans `geocities.csv` pour éviter de
solliciter l'API à chaque exécution.

In [4]:
def geocode_cities(cities):
    """
    Geolocalise une liste de villes en utilisant l'API Nominatim.
    
    Args:
        cities (list): Liste des noms de villes à géocoder
        
    Returns:
        list: Liste de dictionnaires contenant les noms de villes, leurs latitudes, longitudes et types d'adresse
    """

    geocode_api_base = "https://nominatim.openstreetmap.org"

    headers = {
        'User-Agent': USER_AGENT
    }

    # init de la liste où on ajoutera les données des villes, un dict {city, lat, lon, address_type}
    list_geocities = []

    for city in cities:

        try:
            # on utilise le service de recherche en mode forme libre pour permettre une recherche sur une ville, une adresse, un point d'interet
            r = requests.get( f"{geocode_api_base}/search?q={city},France&format=json", headers=headers)
            r.raise_for_status()

            r_cities = r.json()
            # la réponse est une liste on récupère le premier élément
            if len(r_cities) > 0:
                data_city = r_cities[0]
                # récupération des coord GPS
                dict_city = {
                    "city": city,
                    "lat": float(data_city['lat']),
                    "lon": float(data_city['lon']),
                    "address_type": data_city['addresstype']
                }
                print(dict_city)
                list_geocities.append(dict_city)

            else:
                print(f"Pas de données pour la ville: {city}")

        except Exception as e:
            print(f"Erreur de geolocalisation pour la ville {city}: {e}")

        # temporisation d'1 seconde pour limiter les requetes à 1 par seconde
        sleep(1)
    
    return list_geocities

Si les villes ont déjà été géolocalisées, on recharge les données depuis le fichier `geocities.csv`.  
Pour relancer la géolocalisation, depuis l'API, supprimer le fichier `geocities.csv`.

In [5]:
# nettoie les connexions en attente avant toute requête
engine.dispose()

df_cities = pd.read_sql("SELECT * FROM cities", engine)
missing = set(cities) - set(df_cities['city'])

print(f"Villes manquantes: {len(missing)}")

if missing:
    list_geocities = geocode_cities(list(missing))
    df_new = pd.DataFrame(list_geocities)
    # renommage pour correspondre au schéma de la table (lat/lon -> latitude/longitude)
    df_new = df_new.rename(columns={'lat': 'latitude', 'lon': 'longitude'})

    # écriture : toujours dans engine.begin()
    try:
        with engine.begin() as conn:
            df_new.to_sql('cities', conn, if_exists='append', index=False)
        print("Insertion réussie")
    except Exception as e:
        print(f"Erreur insertion: {e}")

    df_cities = pd.read_sql("SELECT * FROM cities", engine)

list_geocities = df_cities.to_dict('records')
print(f"{len(df_cities)} villes en base")

Villes manquantes: 0
35 villes en base


## 3. Récupération données Meteo

L'API Open-Meteo est libre d'accès et gratuite. On utilise comme indicateur de temps, le paramètre`weather_code` qui suit la norme WMO :
0 = ciel dégagé, valeurs croissantes vers la pluie et les orages.  
On le ramène sur une échelle 0–10 (division par 10 au-dessus de 10) puis on calcule la médiane et la moyenne sur 7 jours pour obtenir un indicateur hebdomadaire par ville.

In [6]:
def fetch_cities_weather(list_geocities):
    """
    Récupère les données météo pour une liste de villes
    Calcul un indicateur de meteo moyen sur la semaine pour chaque ville

    Args:
        list_geocities: liste de dictionnaires avec les clés 'city', 'city_id', 'latitude', 'longitude'
    """

    weather_api_url = "https://api.open-meteo.com/v1"

    df_weather_cities = pd.DataFrame()

    for city in list_geocities:
        params = {
            "latitude": city['latitude'],
            "longitude": city['longitude'],
            "daily": "weather_code,temperature_2m_min,temperature_2m_max,precipitation_sum,precipitation_probability_mean",
            "hourly": "weather_code",
            "timezone": "Europe/Berlin"
        }

        try:
            r = requests.get(f"{weather_api_url}/forecast", params=params)
            r.raise_for_status()

            weather_data = r.json()

            # traitement des données daily
            weather_data_d = weather_data['daily']
            # transposition des données en liste de dict
            list_weather_d = [dict(zip(weather_data_d.keys(), values)) for values in zip(*weather_data_d.values())]
            # création dataframe
            df_weather_d = pd.DataFrame(list_weather_d)
            df_weather_d['city'] = city['city']

            # traitement des données hourly
            weather_data_h = weather_data['hourly']
            list_weather_h = list(zip(weather_data_h['time'],weather_data_h['weather_code']))
            # creation dataFrme données hourly pour aggregation daily
            df_weather_h = pd.DataFrame(list_weather_h, columns = ['datetime','weather_code'])
            df_weather_h['datetime'] = pd.to_datetime(df_weather_h['datetime'])

            # normalisation weather_code (de 0 à 10)
            df_weather_h['weather_code'] = df_weather_h['weather_code'].apply(lambda x: round(x / 10) if x >= 10 else x)

            # moyenne et médiane journalière du weather_code
            df_daily_agg = df_weather_h.groupby(df_weather_h['datetime'].dt.day).agg(
                {'weather_code': ['mean', 'median']}).reset_index()

            df_weather_d['weather_code_mean'] = df_daily_agg['weather_code']['mean']
            df_weather_d['weather_code_median'] = df_daily_agg['weather_code']['median']

            # agrégation sur la semaine
            dict_weather_w = {
                'city_id': city['city_id'],
                'city': city['city'],
                'latitude': city['latitude'],
                'longitude': city['longitude'],
                'weather_code_mean': round(df_weather_d['weather_code_mean'].mean(), 2),
                'weather_code_median': df_weather_d['weather_code_median'].median(),
                'temperature_min_mean': round(df_weather_d['temperature_2m_min'].mean(), 1),
                'temperature_max_mean': round(df_weather_d['temperature_2m_max'].mean(), 1),
                'precipitation_sum': df_weather_d['precipitation_sum'].sum()
            }
            df_weather_cities = pd.concat([df_weather_cities, pd.DataFrame([dict_weather_w])], ignore_index=True)

            print(f"Récupération météo pour {city['city']} OK")
            sleep(1)

        except Exception as e:
            print(f"Erreur API Meteo pour {city['city']}: {e}")

    # tri par indicateur weather_code moyen sur la semaine
    df_weather_cities.sort_values("weather_code_mean", inplace=True)
    return df_weather_cities

In [ ]:
today = date.today()

# villes sans données météo pour aujourd'hui
df_weather_cities = pd.read_sql(
    "SELECT city_id FROM weather_cities WHERE date = %(d)s",
    engine, params={"d": today}
)
missing_ids = set(df_cities['city_id']) - set(df_weather_cities['city_id'])
missing_cities = [c for c in list_geocities if c['city_id'] in missing_ids]

if missing_cities:
    # appel API uniquement pour les villes manquantes
    df_weather_new = fetch_cities_weather(missing_cities)

    # sélection des colonnes correspondant au schéma de la table
    df_to_insert = df_weather_new[[
        'city_id', 'weather_code_mean',
        'temperature_min_mean', 'temperature_max_mean', 'precipitation_sum'
    ]].copy()
    df_to_insert['date'] = today

    try:
        with engine.begin() as conn:
            df_to_insert.to_sql('weather_cities', conn, if_exists='append', index=False)
        print(f"{len(df_to_insert)} lignes insérées dans weather_cities")
    except Exception as e:
        print(f"Erreur insertion dans weather_cities: {e}")
else:
    print("Données météo du jour déjà en base")


Données météo du jour déjà en base
